# RG3 가스 탐색 후보 → 비라벨 적용
정답 없는 추론이다. 가스 후보와 확정 가스를 구분한다. 기존 17가스·566정상으로 최종 재학습하고 OOF 정상 경계를 고정한다.

In [1]:
from pathlib import Path
import json, hashlib, numpy as np, pandas as pd
from sklearn.metrics import confusion_matrix
root=Path.cwd()
if not (root/'data').exists():root=root.parent
tables=root/'reports/tables'
def notebook_source(name):
 return ''.join(json.loads((root/'notebooks'/name).read_text(encoding='utf-8'))['cells'][1]['source'])
# Rebuild the same sensor history as notebook10, with both products preserving equipment switches.
pre={'__name__':'__main__'}
s10=notebook_source('10_original_defect_report_revision.ipynb')
exec(s10.split('shots = label_shots.merge')[0],pre)
observed=pre['observed'];keys=pre['keys'];raw_features=pre['features']
saved=pd.read_csv(tables/'original_labeled_shot_features.csv',parse_dates=['TimeStamp'])
joined=saved.merge(observed,on=keys,suffixes=('_saved','_rebuilt'),validate='one_to_one')
compare=[c for c in saved if c in observed and c not in keys+['history_segment','sensor_conflict']]
for c in compare:
 a=joined[c+'_saved'].to_numpy(float);b=joined[c+'_rebuilt'].to_numpy(float)
 assert np.allclose(a,b,rtol=1e-8,atol=1e-7,equal_nan=True),c
print('Same historical feature construction verified on all 1304 labeled shots',flush=True)
u=pd.read_csv(root/'data/original_split/moldset_unlabeled_rg3.csv',parse_dates=['TimeStamp'])
ukeys=pd.MultiIndex.from_frame(u.assign(product='RG3')[keys]).unique()
cohort=pd.read_csv(root/'data/original_split/moldset_labeled_rg3.csv',parse_dates=['TimeStamp'])
knownkeys=pd.MultiIndex.from_frame(cohort.assign(product='RG3')[keys]).unique()
# Also exclude keys in any supplied raw labeled file, without using their quality labels.
extra=pd.read_csv(root/'data/original/labeled_data.csv',usecols=['EQUIP_CD','PART_NAME','TimeStamp'],parse_dates=['TimeStamp'])
extra=extra[extra.PART_NAME.str.contains('RG3',na=False)].assign(product='RG3')
allknown=knownkeys.union(pd.MultiIndex.from_frame(extra[keys]).unique())
sel=pd.MultiIndex.from_frame(observed[keys]).isin(ukeys)
qtable=observed[sel & observed['product'].eq('RG3')].copy()
qtable['known_label_overlap']=pd.MultiIndex.from_frame(qtable[keys]).isin(allknown)
overlap=int(qtable.known_label_overlap.sum())
qtable=qtable[~qtable.known_label_overlap].reset_index(drop=True)
print('Raw unlabeled',len(u),'unique keys',len(ukeys),'overlap excluded',overlap,'remaining',len(qtable),flush=True)
base={'__name__':'__main__'}
exec(notebook_source('22_rg3_nested_signal_ensemble.ipynb').split('scores=[];thresholds=[];foldlogs=[]')[0].replace('n_jobs=-1','n_jobs=2'),base)
train=base['d'].copy();n=len(train)
assert n==583 and train.PassOrFail.sum()==17
qtable=qtable[qtable.EQUIP_CD.isin(train.EQUIP_CD.unique())].reset_index(drop=True)
# Unknown query labels are -1, never pseudo-normal; only ref indices enter supervised fit.
qd=qtable.reindex(columns=train.columns).copy()
qd['comparison_block']='UNLABELED_'+qtable.history_segment.astype(str)
qd['PassOrFail']=-1;qd['Reason']='UNKNOWN'
# Query-only median fill permits scoring; incomplete/conflicting rows remain abstentions below.
qd[raw_features]=qd[raw_features].fillna(train.loc[train.PassOrFail.eq(0),raw_features].median())
base['d']=pd.concat([train,qd],ignore_index=True)
base['y']=np.r_[train.PassOrFail.to_numpy(int),np.full(len(qd),-1)]
base['groups']=base['d'].comparison_block
ref=np.arange(n);query=np.arange(n,n+len(qd))
assert set(base['d'].TimeStamp.iloc[ref]).isdisjoint(base['d'].TimeStamp.iloc[query])
# Final-fit baseline, threshold from leave-one-block-out normal predictions.
bp=base['pair_predict'](ref,query).max(axis=1)
cal=pd.read_csv(tables/'rg3_nested_ensemble_predictions.csv',parse_dates=['TimeStamp'])
cal=cal[cal.scheme.eq('leave_one_block_out') & cal.model.eq('max_union')].set_index('TimeStamp').loc[train.TimeStamp]
cb=float(np.quantile(cal.loc[train.PassOrFail.to_numpy()==0,'score'],.99,method='higher'))
baseline=bp>cb
print('Baseline scored',len(query),flush=True)
# Same 47 three-seed gas-mode experts; extend its [7,101] seeds to include seed42.
from sklearn.cluster import KMeans
base['KMeans']=KMeans
s47=notebook_source('47_rg3_innovation_seed_sensitivity.ipynb')
definition=s47[s47.index('def expert('):s47.index('records=[];audit=[];rawrecords=[]')]
definition=definition.replace('scores=[[],[]]','scores=[[],[],[]]').replace('enumerate([7,101])','enumerate([42,7,101])')
base['memo']={}
exec(definition,base)
ex=base['expert'](ref,query).mean(axis=1)
oldex=pd.read_csv(tables/'rg3_innovation_seed_sensitivity_raw_scores.csv',parse_dates=['TimeStamp'])
oldex=oldex[oldex.scheme.eq('leave_one_block_out')].set_index('TimeStamp').loc[train.TimeStamp]
excal=oldex[['seed42','seed7','seed101']].mean(axis=1).to_numpy()
normal=train.PassOrFail.eq(0).to_numpy()
ce=float(np.quantile(excal[normal],.995,method='higher'))
delta=train.Barrel_Temperature_5__delta1.to_numpy(float)
cd=float(np.quantile(delta[normal & np.isfinite(delta)],.95,method='higher'))
innovation=(ex>ce)&(qd.Barrel_Temperature_5__delta1.to_numpy()>cd)
anchor47=baseline|innovation
print('Innovation scored',flush=True)
# Reuse exact 54 thermal feature construction and local-residual/predict functions without running its evaluation.
s54=notebook_source('54_rg3_pdf_thermal_confirm_gates.ipynb')
start=s54.index('# Mechanism names');end=s54.index("models=['two_tree_mean")
# Required imports are already in base except ExtraTrees and nearest-neighbor.
from sklearn.ensemble import ExtraTreesClassifier
from sklearn.neighbors import NearestNeighbors
base['ExtraTreesClassifier']=ExtraTreesClassifier;base['NearestNeighbors']=NearestNeighbors
thermal_setup=s54[start:end]
write_start=thermal_setup.index('for name,x in F.items():');write_end=thermal_setup.index('from sklearn.neighbors import NearestNeighbors')
thermal_setup=thermal_setup[:write_start]+thermal_setup[write_end:]
exec(thermal_setup,base)
local_predict_source=s54[s54.index('def predict('):end]
therm_name='surface_thermal_contact'
thermal=base['predict'](ref,query,therm_name)[:,1]
thermal_z=base['local_residual'](ref,query,therm_name)[:,4]
tcal=pd.read_csv(tables/'rg3_pdf_confirm_gates_predictions.csv',parse_dates=['TimeStamp'])
tcal=tcal[tcal.scheme.eq('leave_one_block_out')&tcal.model.eq('thermal_change_low_0.05')].set_index('TimeStamp').loc[train.TimeStamp]
ct=float(np.quantile(tcal.loc[normal,'score'],.975,method='higher'))
cg=float(np.quantile(tcal.loc[normal,'gate_value_thermal'],.05))
thermal_add=(thermal>ct)&(thermal_z<cg)
anchor54=anchor47|thermal_add
print('Thermal anchor scored',flush=True)
# Exact pressure_transition feature expression from 58.
d=base['d']
base['F']['pressure_transition']=pd.DataFrame({
 'pressure_minus_switch_delta':d.Max_Injection_Pressure__delta1-d.Max_Switch_Over_Pressure__delta1,
 'speed_delta':d.Max_Injection_Speed__delta1,
 'filling_delta':d.Filling_Time__delta1,
 'pressure_minus_switch_dev10':d.Max_Injection_Pressure__dev10-d.Max_Switch_Over_Pressure__dev10,
 'cushion_delta':d.Cushion_Position__delta1,
 'barrel_mold_delta_gap':d.Barrel_Temperature_5__delta1-(d.Mold_Temperature_3__delta1+d.Mold_Temperature_4__delta1)/2}).replace([np.inf,-np.inf],np.nan)
base['controls']['pressure_transition']=['Max_Injection_Speed','Filling_Time','Barrel_Temperature_5','Mold_Temperature_3']
direction=(qd.Max_Injection_Pressure__delta1.round(2)<0)&(qd.Max_Injection_Speed__delta1.round(2)>0)&(qd.Max_Switch_Over_Pressure__delta1.round(2)<qd.Max_Injection_Pressure__delta1.round(2))
pressure=[];thresholds=[]
for tag,seeds,file in [
 ('42_7_101',[42,7,101],'rg3_pressure_direction_predictions.csv'),
 ('11_23_37',[11,23,37],'rg3_pressure_seed_seed11_23_37_predictions.csv'),
 ('13_29_53',[13,29,53],'rg3_pressure_seed_seed13_29_53_predictions.csv')]:
 exec(local_predict_source.replace('for seed in [42,7,101]:',f'for seed in {seeds}:'),base);base['memo']={}
 score=base['predict'](ref,query,'pressure_transition')[:,1]
 pc=pd.read_csv(tables/file,parse_dates=['TimeStamp'])
 pc=pc[pc.scheme.eq('leave_one_block_out')&pc.anchor.eq('exploratory54')&pc.model.eq('pressure_fall_speed_rise_switch_steeper')].set_index('TimeStamp').loc[train.TimeStamp]
 cut=float(np.quantile(pc.loc[normal,'score'],.99,method='higher'))
 pressure.append((score>cut)&direction.to_numpy())
 thresholds.append(dict(branch='pressure_'+tag,threshold=cut))
 qtable['pressure_score_'+tag]=score
 print('Pressure seed bundle',tag,'scored',flush=True)
votes=np.column_stack(pressure).sum(axis=1);pressure_add=votes>=2;final=anchor54|pressure_add
required=list(dict.fromkeys(raw_features+base['stdcols']+[
 'Barrel_Temperature_5__delta1','Mold_Temperature_3__delta1','Mold_Temperature_4__delta1','Mold_Temperature_3__dev10','Mold_Temperature_4__dev10','Max_Injection_Pressure__delta1','Max_Switch_Over_Pressure__delta1','Max_Injection_Speed__delta1','Filling_Time__delta1','Cushion_Position__delta1','Max_Injection_Pressure__dev10','Max_Switch_Over_Pressure__dev10']))
ready=np.isfinite(qtable[required].to_numpy(float)).all(axis=1)&~qtable.sensor_conflict
out=qtable[keys+['history_segment','sensor_conflict']].copy()
out['history_ready']=ready;out['raw_candidate']=final.astype(int)
out['gas_candidate']=np.where(ready,final.astype(float),np.nan)
out['baseline_candidate']=baseline.astype(int);out['innovation_add']=innovation.astype(int);out['thermal_add']=thermal_add.astype(int);out['pressure_add']=pressure_add.astype(int);out['pressure_votes']=votes
out['baseline_score']=bp;out['expert_score']=ex;out['thermal_score']=thermal;out['thermal_gate_value']=thermal_z
oodcols=['Max_Injection_Speed','Filling_Time','Barrel_Temperature_5','Mold_Temperature_3','Barrel_Temperature_2','Hopper_Temperature','Cushion_Position','Average_Screw_RPM']
lo=train.loc[normal,oodcols].quantile(.01);hi=train.loc[normal,oodcols].quantile(.99)
out['outside_normal_1_99_count']=((qd[oodcols]<lo)|(qd[oodcols]>hi)).sum(axis=1)
out['date']=out.TimeStamp.dt.strftime('%Y-%m-%d')
for c in raw_features:out[c]=qtable[c]
out.to_csv(tables/'rg3_unlabeled_gas_candidate_predictions.csv',index=False,encoding='utf-8-sig')
out[out.gas_candidate.eq(1)].to_csv(tables/'rg3_unlabeled_gas_candidates_only.csv',index=False,encoding='utf-8-sig')
byday=out.groupby('date').agg(shots=('TimeStamp','size'),ready=('history_ready','sum'),gas_candidates=('gas_candidate','sum'))
byday['candidate_rate_ready']=byday.gas_candidates/byday.ready.replace(0,np.nan)
byday.to_csv(tables/'rg3_unlabeled_gas_candidates_by_day.csv',encoding='utf-8-sig')
byseg=out.groupby('history_segment').agg(start=('TimeStamp','min'),end=('TimeStamp','max'),shots=('TimeStamp','size'),ready=('history_ready','sum'),gas_candidates=('gas_candidate','sum'))
byseg['candidate_rate_ready']=byseg.gas_candidates/byseg.ready.replace(0,np.nan)
byseg.to_csv(tables/'rg3_unlabeled_gas_candidates_by_segment.csv',encoding='utf-8-sig')
branches=[]
for name,pr in [('baseline',baseline),('innovation',anchor47),('thermal_anchor',anchor54),('pressure_majority_final',final)]:
 branches.append(dict(branch=name,ready_shots=int(ready.sum()),candidates=int((pr&ready).sum()),candidate_rate=float((pr&ready).sum()/ready.sum())))
pd.DataFrame(branches).to_csv(tables/'rg3_unlabeled_gas_branch_counts.csv',index=False,encoding='utf-8-sig')
thresholds+= [dict(branch='baseline',threshold=cb),dict(branch='expert',threshold=ce),dict(branch='barrel5_delta',threshold=cd),dict(branch='thermal_score',threshold=ct),dict(branch='thermal_gate',threshold=cg)]
pd.DataFrame(thresholds).to_csv(tables/'rg3_unlabeled_gas_frozen_thresholds.csv',index=False,encoding='utf-8-sig')
# Agreement of full-refit candidate inputs with saved training data audited, not a new validation.
summary=dict(unlabeled_rows=len(u),unlabeled_unique_keys=len(ukeys),known_overlap_excluded=overlap,query_shots=len(out),history_ready=int(ready.sum()),history_insufficient=int((~ready).sum()),gas_candidates=int((final&ready).sum()),train_normal=566,train_gas=17,query_quality_used=0,train_query_key_overlap=0,feature_rebuild_match=True,query_before_train_end=int((out.TimeStamp<train.TimeStamp.max()).sum()))
(tables/'rg3_unlabeled_gas_inference_audit.json').write_text(json.dumps(summary,ensure_ascii=False,indent=2),encoding='utf-8')
report='# RG3 비라벨 데이터: 기존 가스 후보 모델 적용\n\n'
report+='8TP/6FP 탐색 후보의 피처·모델·분기 조건을 유지해 기존 정상566/가스17로 최종 재학습한 적용본. 기존 구간별 평가 모델을 그대로 저장해 사용한 것은 아니다. 임계값은 기존 라벨 구간제외 OOF 정상 점수로 계산; 비라벨 분포나 개수로 맞추지 않았다. 추가 가스19 코호트·비라벨 의사정답은 학습에 사용하지 않았다.\n\n'
report+='비라벨 RH/LH 동일 설비·제품·Timestamp를 Shot으로 통합. 모든 알려진 라벨 키 중복 제외. 10번과 같은 두 제품 전체 이력, 설비 제품 전환/30분 공백, shift1 과거10개 최소5개 피처를 재구성하고 기존1304Shot 입력과 일치 확인. 센서충돌 또는 필요한 이력 미비는 정상 판정하지 않고 판정보류.\n\n'
report+=json.dumps(summary,ensure_ascii=False,indent=2)+'\n\n'+pd.DataFrame(branches).to_csv(index=False)
report+='\n\n**가스 후보는 확정 불량이 아니다. 정답이 없어 TP/FP/F1을 계산할 수 없다.** 과거10/21~23 라벨로 그 이전 날짜까지 소급 추론하므로 실시간 미래 독립검증이 아니다. 비라벨 센서는 기존 파생피처 이력에도 쓰였으므로 완전히 새로운 입력 분포 시험도 아니다.\n\n'
report+='학습 정상1~99% 범위를 벗어난 제어센서 수는 분포 차이 진단용이며 판정 임계값 조정에 쓰지 않았다. history_segment는 연속 관측 후보 구간이지 실제 설비 Run ID가 아니다.\n\n'
report+='## 날짜별 결과\n'+byday.to_csv()+'\n## 탐지율 높은 연속 관측 후보구간\n'+byseg[byseg.ready>=20].sort_values('candidate_rate_ready',ascending=False).head(15).to_csv()
(root/'reports/rg3_unlabeled_gas_candidate_review.md').write_text(report,encoding='utf-8')
print('SUMMARY',json.dumps(summary,ensure_ascii=False),flush=True)
print(pd.DataFrame(branches).to_string(index=False),flush=True)
print(byday.to_string(),flush=True)
print('Top segments',byseg[byseg.ready>=20].sort_values('candidate_rate_ready',ascending=False).head(10).to_string(),flush=True)


Same historical feature construction verified on all 1304 labeled shots


Raw unlabeled 72484 unique keys 66145 overlap excluded 387 remaining 65748


Baseline scored 29236


Innovation scored


Thermal anchor scored


...

Pressure seed bundle 13_29_53 scored


SUMMARY {"unlabeled_rows": 72484, "unlabeled_unique_keys": 66145, "known_overlap_excluded": 387, "query_shots": 29236, "history_ready": 28710, "history_insufficient": 526, "gas_candidates": 287, "train_normal": 566, "train_gas": 17, "query_quality_used": 0, "train_query_key_overlap": 0, "feature_rebuild_match": true, "query_before_train_end": 29236}


                 branch  ready_shots  candidates  candidate_rate
               baseline        28710         174        0.006061
             innovation        28710         176        0.006130
         thermal_anchor        28710         264        0.009195
pressure_majority_final        28710         287        0.009997


            shots  ready  gas_candidates  candidate_rate_ready
date                                                          
2020-03-25      1      0             0.0                   NaN
2020-03-26     22      0             0.0                   NaN
2020-03-27     22      0             0.0                   NaN
2020-03-28    307    290             1.0              0.003448
2020-03-29   1788   1768             7.0              0.003959
2020-03-30    495    485            13.0              0.026804
2020-03-31   2755   2691             2.0              0.000743
2020-04-20     12      0             0.0                   NaN
2020-04-21     72     64             0.0              0.000000
2020-04-22      2      0             0.0                   NaN
2020-04-23     53     30             0.0              0.000000
2020-04-24    549    544             1.0              0.001838
2020-04-25    101     91             0.0              0.000000
2020-04-27    666    655            21.0              0

Top segments                               start                 end  shots  ready  gas_candidates  candidate_rate_ready
history_segment                                                                                            
259             2020-04-27 18:02:57 2020-04-27 19:07:11     40     35            13.0              0.371429
345             2020-07-20 01:22:04 2020-07-20 02:08:28     39     34             4.0              0.117647
308             2020-06-23 14:11:21 2020-06-23 23:23:34    493    488            24.0              0.049180
504             2020-10-20 09:22:18 2020-10-22 23:24:42   3234   3229           141.0              0.043667
341             2020-07-15 23:16:49 2020-07-16 23:30:02   1348   1343            57.0              0.042442
309             2020-07-03 02:32:17 2020-07-03 04:32:14    114    109             2.0              0.018349
325             2020-07-14 15:09:03 2020-07-15 00:36:27    471    466             8.0              0.017167
258            